# 🏆 HackMate
Your AI copilot for hackathons. Run all cells — a menu appears at the bottom to choose your tool.

---

In [8]:
!pip install groq ipywidgets


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: C:\Users\varun\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [ ]:
import os
import json
import re
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from groq import Groq

os.environ["GROQ_API_KEY"] = "........"  # Replace with your key
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [3]:
JUDGE_SYSTEM_PROMPT = "You are an experienced hackathon judge with deep expertise across technical, business, and social impact domains. You adapt your judging style based on project type.\n\nYou evaluate projects across:\n- Innovation & Creativity\n- Technical Execution\n- Impact & Feasibility\n- Demo & Communication\n\n## Your Persona\nSharp but fair. One focused question at a time. You push back on vague answers. You immediately spot ChatGPT wrappers, over-scoped ideas, and genuinely novel work.\n\n## Adapt by Project Type\n- AI/ML: differentiation from existing tools, hallucination risks, data quality\n- Social impact: real user research, sustainability, measurable outcomes\n- Dev tools: workflow fit, existing alternatives, adoption friction\n- Consumer apps: retention, market size, monetization\n- Fintech/Healthcare: compliance, privacy, liability\n\n## Style\nAsk ONE specific question at a time. After 4-6 exchanges, give a final scorecard.\n\n## Final Scorecard Format\n```json\n{\n  \"scores\": {\n    \"innovation\": 8,\n    \"technical_execution\": 6,\n    \"impact_feasibility\": 7,\n    \"communication\": 9\n  },\n  \"overall\": 7.5,\n  \"strengths\": [\"strength 1\", \"strength 2\"],\n  \"critical_feedback\": [\"feedback 1\", \"feedback 2\"],\n  \"one_thing_to_fix\": \"Single most important improvement\"\n}\n```"

In [4]:
CHECKLIST_PROMPT = "You are a senior hackathon mentor reviewing a team's Devpost submission before they submit. You have judged and mentored at dozens of MLH hackathons.\n\nYour job is to review their project description and README, then produce a structured pre-submission checklist report.\n\nYou evaluate across these categories:\n\n1. DEVPOST DESCRIPTION\n   - Is the problem statement clear and compelling?\n   - Is the solution explained in plain English (not just tech jargon)?\n   - Is there a clear target user?\n   - Does it mention the tech stack?\n   - Is there a \"what's next\" / future plans section?\n\n2. README QUALITY\n   - Does it have setup/installation instructions?\n   - Is there a demo link or screenshots?\n   - Is the architecture or technical approach explained?\n   - Are team members listed?\n\n3. PITCH CLARITY\n   - Can a non-technical judge understand the value in 30 seconds?\n   - Is the innovation clearly stated \u2014 what makes this different?\n   - Is there a concrete use case or demo flow described?\n\n4. RED FLAGS\n   - Vague problem statements\n   - No demo or screenshots\n   - Overly technical with no user story\n   - Missing setup instructions\n   - No mention of what was built during the hackathon vs pre-existing code\n\nRespond ONLY with a JSON block in this exact format:\n```json\n{\n  \"scores\": {\n    \"devpost_description\": 7,\n    \"readme_quality\": 5,\n    \"pitch_clarity\": 8,\n    \"red_flags\": 2\n  },\n  \"overall_readiness\": \"STRONG\",\n  \"checklist\": {\n    \"devpost_description\": [\n      {\"item\": \"Problem statement is clear\", \"status\": \"pass\"},\n      {\"item\": \"Solution explained in plain English\", \"status\": \"fail\"},\n      {\"item\": \"Target user identified\", \"status\": \"pass\"},\n      {\"item\": \"Tech stack mentioned\", \"status\": \"pass\"},\n      {\"item\": \"Future plans included\", \"status\": \"warning\"}\n    ],\n    \"readme_quality\": [\n      {\"item\": \"Setup/installation instructions\", \"status\": \"fail\"},\n      {\"item\": \"Demo link or screenshots\", \"status\": \"pass\"},\n      {\"item\": \"Architecture explained\", \"status\": \"warning\"},\n      {\"item\": \"Team members listed\", \"status\": \"pass\"}\n    ],\n    \"pitch_clarity\": [\n      {\"item\": \"Non-technical judge can understand value\", \"status\": \"pass\"},\n      {\"item\": \"Innovation clearly stated\", \"status\": \"warning\"},\n      {\"item\": \"Concrete use case or demo flow\", \"status\": \"pass\"}\n    ]\n  },\n  \"red_flags\": [\"flag 1 if any\", \"flag 2 if any\"],\n  \"top_3_fixes\": [\n    \"Most important fix before submitting\",\n    \"Second most important fix\",\n    \"Third most important fix\"\n  ],\n  \"one_liner_suggestion\": \"A suggested one-line project description that would hook a judge\"\n}\n```\noverall_readiness must be one of: STRONG, GOOD, NEEDS_WORK, NOT_READY\nred_flags score is how many red flags found (lower is better).\nstatus values: pass, fail, warning"

print('✅ Checklist prompt loaded.')

✅ Checklist prompt loaded.


In [5]:
def parse_score_block(text):
    match = re.search(r'''```json\s*(.*?)\s*```''', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            return None
    return None

def format_scores_html(data):
    scores = data.get("scores", {})
    labels = {
        "innovation": "Innovation & Creativity",
        "technical_execution": "Technical Execution",
        "impact_feasibility": "Impact & Feasibility",
        "communication": "Communication & Demo"
    }
    html = "<div style='font-family:monospace; background:#1a1a2e; color:#eee; padding:20px; border-radius:8px; margin:10px 0'>"
    html += "<h3 style='color:#f0c040; margin-top:0'>🏆 FINAL JUDGE SCORECARD</h3>"
    for key, label in labels.items():
        score = scores.get(key, 0)
        pct = int(score) * 10
        html += f"<div style='margin:8px 0'><span style='display:inline-block;width:220px'>{label}</span>"
        html += f"<div style='display:inline-block;width:200px;background:#333;border-radius:4px;height:16px;vertical-align:middle'>"
        html += f"<div style='width:{pct}%;background:#f0c040;height:100%;border-radius:4px'></div></div>"
        html += f" <b>{score}/10</b></div>"
    overall = data.get("overall", "N/A")
    html += f"<h3 style='color:#f0c040'>OVERALL: {overall}/10</h3>"
    html += "<h4 style='color:#4caf50'>✅ Strengths</h4><ul>"
    for s in data.get("strengths", []):
        html += f"<li>{s}</li>"
    html += "</ul><h4 style='color:#ff7043'>⚠️ Critical Feedback</h4><ul>"
    for f in data.get("critical_feedback", []):
        html += f"<li>{f}</li>"
    html += "</ul>"
    fix = data.get("one_thing_to_fix", "")
    if fix:
        html += f"<h4 style='color:#29b6f6'>🎯 Most Important Fix</h4><p>{fix}</p>"
    html += "</div>"
    return html

In [6]:
def format_checklist_html(data):
    readiness = data.get("overall_readiness", "UNKNOWN")
    colors = {"STRONG": "#4caf50", "GOOD": "#8bc34a", "NEEDS_WORK": "#ff9800", "NOT_READY": "#f44336"}
    color = colors.get(readiness, "#888")

    html = f"<div style='font-family:sans-serif;background:#1a1a2e;color:#eee;padding:20px;border-radius:8px;margin:10px 0'>"
    html += f"<h3 style='color:#f0c040;margin-top:0'>📋 Pre-Submission Report</h3>"
    html += f"<div style='background:{color};color:#000;padding:8px 16px;border-radius:4px;display:inline-block;font-weight:bold;margin-bottom:16px'>Overall Readiness: {readiness}</div>"

    # Scores
    scores = data.get("scores", {})
    score_labels = {
        "devpost_description": "Devpost Description",
        "readme_quality": "README Quality",
        "pitch_clarity": "Pitch Clarity"
    }
    html += "<h4 style='color:#f0c040'>Scores</h4>"
    for key, label in score_labels.items():
        score = scores.get(key, 0)
        pct = int(score) * 10
        html += f"<div style='margin:6px 0'><span style='display:inline-block;width:200px'>{label}</span>"
        html += f"<div style='display:inline-block;width:180px;background:#333;border-radius:4px;height:14px;vertical-align:middle'>"
        html += f"<div style='width:{pct}%;background:#f0c040;height:100%;border-radius:4px'></div></div>"
        html += f" <b>{score}/10</b></div>"

    # Checklist
    status_icons = {"pass": "✅", "fail": "❌", "warning": "⚠️"}
    checklist = data.get("checklist", {})
    section_labels = {
        "devpost_description": "Devpost Description",
        "readme_quality": "README",
        "pitch_clarity": "Pitch Clarity"
    }
    html += "<h4 style='color:#f0c040;margin-top:16px'>Checklist</h4>"
    for section, label in section_labels.items():
        items = checklist.get(section, [])
        if items:
            html += f"<p style='color:#aaa;margin:8px 0 4px'><b>{label}</b></p>"
            for item in items:
                icon = status_icons.get(item.get("status", "warning"), "⚠️")
                html += f"<div style='margin:3px 0'>{icon} {item.get('item', '')}</div>"

    # Red flags
    red_flags = data.get("red_flags", [])
    if red_flags:
        html += "<h4 style='color:#f44336;margin-top:16px'>🚩 Red Flags</h4><ul>"
        for f in red_flags:
            html += f"<li>{f}</li>"
        html += "</ul>"

    # Top 3 fixes
    fixes = data.get("top_3_fixes", [])
    if fixes:
        html += "<h4 style='color:#29b6f6;margin-top:16px'>🎯 Top 3 Fixes Before Submitting</h4><ol>"
        for fix in fixes:
            html += f"<li style='margin:4px 0'>{fix}</li>"
        html += "</ol>"

    # One liner
    one_liner = data.get("one_liner_suggestion", "")
    if one_liner:
        html += f"<h4 style='color:#ce93d8'>💬 Suggested Hook Line</h4>"
        html += f"<div style='background:#2a2a3e;padding:12px;border-radius:4px;font-style:italic'>{one_liner}</div>"

    html += "</div>"
    return html

print("✅ Checklist helpers loaded.")

✅ Checklist helpers loaded.


In [7]:
# ═══════════════════════════════════════════════
# HACKMATE LAUNCHER — defines all UI, shows menu
# ═══════════════════════════════════════════════

# ── JUDGE SIMULATOR WIDGETS ──────────────────
# State
state = {"history": [], "exchange_count": 0}

# UI elements
project_input = widgets.Textarea(
    placeholder="Describe your project: what problem you solve, what you built, your stack, who it's for...",
    layout=widgets.Layout(width="100%", height="100px")
)
start_btn = widgets.Button(description="Start Judging", button_style="success",
                           layout=widgets.Layout(width="150px"))
answer_input = widgets.Text(
    placeholder="Type your answer and press Enter or click Send...",
    layout=widgets.Layout(width="80%", display="none")
)
send_btn = widgets.Button(description="Send", button_style="primary",
                          layout=widgets.Layout(width="100px", display="none"))
chat_output = widgets.Output()

MAX_QUESTIONS = 5

def add_message(role, text):
    with chat_output:
        if role == "judge":
            display(HTML(f"<div style='background:#1e1e2e;color:#cdd6f4;padding:12px 16px;border-left:4px solid #f0c040;border-radius:4px;margin:8px 0;font-family:sans-serif'><b style='color:#f0c040'>⚖️ Judge</b><br>{text}</div>"))
        else:
            display(HTML(f"<div style='background:#2a2a3e;color:#cdd6f4;padding:12px 16px;border-left:4px solid #89b4fa;border-radius:4px;margin:8px 0;font-family:sans-serif'><b style='color:#89b4fa'>🎤 You</b><br>{text}</div>"))

def call_judge(messages):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        max_tokens=1024,
        messages=[{"role": "system", "content": JUDGE_SYSTEM_PROMPT}] + messages
    )
    return response.choices[0].message.content

def force_scorecard():
    """Force the model to produce a scorecard with no further questions."""
    messages = state["history"] + [{
        "role": "user",
        "content": "You have asked enough questions. NOW produce the final scorecard in the exact JSON format specified. Do not ask any more questions."
    }]
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        max_tokens=1024,
        messages=[{"role": "system", "content": JUDGE_SYSTEM_PROMPT}] + messages
    )
    return response.choices[0].message.content

def end_session(reply):
    scores_data = parse_score_block(reply)
    if scores_data:
        text_before = reply[:reply.find("```json")].strip()
        if text_before:
            add_message("judge", text_before)
        with chat_output:
            display(HTML(format_scores_html(scores_data)))
    else:
        # Model still didn't return JSON — force it one more time
        with chat_output:
            display(HTML("<div style='color:#f0c040;font-family:sans-serif;padding:8px'>⏳ Generating scorecard...</div>"))
        forced = force_scorecard()
        scores_data = parse_score_block(forced)
        if scores_data:
            with chat_output:
                display(HTML(format_scores_html(scores_data)))
        else:
            add_message("judge", forced)
    answer_input.disabled = True
    send_btn.disabled = True

def on_start(b):
    project = project_input.value.strip()
    if not project:
        with chat_output:
            display(HTML("<p style='color:red'>Please describe your project first.</p>"))
        return
    start_btn.disabled = True
    project_input.disabled = True
    answer_input.layout.display = ""
    send_btn.layout.display = ""
    with chat_output:
        display(HTML("<hr><h4 style='color:#f0c040;font-family:sans-serif'>--- JUDGING SESSION STARTED ---</h4>"))
    initial = f"A team is presenting their hackathon project:\n\n{project}\n\nBriefly acknowledge what they built (1 sentence only), then ask your first sharp judging question. One question only. Do not ask multiple questions."
    state["history"].append({"role": "user", "content": initial})
    reply = call_judge(state["history"])
    state["history"].append({"role": "assistant", "content": reply})
    add_message("judge", reply)

def on_send(b):
    answer = answer_input.value.strip()
    if not answer:
        return
    answer_input.value = ""
    add_message("you", answer)
    state["exchange_count"] += 1

    # Hard stop at MAX_QUESTIONS
    if state["exchange_count"] >= MAX_QUESTIONS:
        state["history"].append({"role": "user", "content": answer})
        reply = force_scorecard()
        end_session(reply)
        return

    state["history"].append({"role": "user", "content": answer})
    reply = call_judge(state["history"])
    state["history"].append({"role": "assistant", "content": reply})

    scores_data = parse_score_block(reply)
    if scores_data:
        end_session(reply)
    else:
        add_message("judge", reply)

answer_input.on_submit(lambda x: on_send(None))
send_btn.on_click(on_send)
start_btn.on_click(on_start)

# ── CHECKLIST WIDGETS ────────────────────────
# Checklist UI
cl_project_input = widgets.Textarea(
    placeholder="Paste your Devpost project description here...",
    layout=widgets.Layout(width="100%", height="120px")
)
cl_readme_input = widgets.Textarea(
    placeholder="Paste your README content here...",
    layout=widgets.Layout(width="100%", height="120px")
)
cl_btn = widgets.Button(description="Review Submission", button_style="warning",
                        layout=widgets.Layout(width="200px"))
cl_output = widgets.Output()

def on_review(b):
    project_text = cl_project_input.value.strip()
    readme_text = cl_readme_input.value.strip()
    if not project_text or not readme_text:
        with cl_output:
            display(HTML("<p style='color:red'>Please paste both your Devpost description and README.</p>"))
        return
    cl_btn.disabled = True
    with cl_output:
        clear_output()
        display(HTML("<p style='color:#f0c040;font-family:sans-serif'>⏳ Reviewing your submission...</p>"))

    prompt = f"""Please review this hackathon submission:

DEVPOST DESCRIPTION:
{project_text}

README:
{readme_text}

Produce the full checklist report in the JSON format specified."""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        max_tokens=2000,
        messages=[
            {"role": "system", "content": CHECKLIST_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )
    reply = response.choices[0].message.content
    data = parse_score_block(reply)

    with cl_output:
        clear_output()
        if data:
            display(HTML(format_checklist_html(data)))
        else:
            display(HTML(f"<pre style='color:#eee;background:#1a1a2e;padding:12px;border-radius:4px'>{reply}</pre>"))
    cl_btn.disabled = False

cl_btn.on_click(on_review)

# ── MAIN MENU ────────────────────────────────
tool1_btn = widgets.Button(
    description="1 — Judge Simulator",
    button_style="success",
    layout=widgets.Layout(width="230px", height="50px")
)
tool2_btn = widgets.Button(
    description="2 — Submission Checklist",
    button_style="warning",
    layout=widgets.Layout(width="230px", height="50px")
)
main_output = widgets.Output()

def show_judge(b):
    # Reset judge state
    state["history"] = []
    state["exchange_count"] = 0
    project_input.value = ""
    project_input.disabled = False
    start_btn.disabled = False
    answer_input.layout.display = "none"
    send_btn.layout.display = "none"
    answer_input.disabled = False
    send_btn.disabled = False
    chat_output.clear_output()
    with main_output:
        clear_output(wait=True)
        display(HTML("""
        <div style='font-family:sans-serif;background:#1a1a2e;padding:16px;border-radius:8px;margin-bottom:12px'>
          <h3 style='color:#f0c040;margin:0 0 4px 0'>⚖️ Judge Simulator</h3>
          <p style='color:#888;margin:0'>Describe your project below. Face the judge. Get scored after 5 questions.</p>
        </div>
        """))
        display(project_input)
        display(start_btn)
        display(chat_output)
        display(widgets.HBox([answer_input, send_btn]))

def show_checklist(b):
    # Reset checklist state
    cl_project_input.value = ""
    cl_readme_input.value = ""
    cl_btn.disabled = False
    cl_output.clear_output()
    with main_output:
        clear_output(wait=True)
        display(HTML("""
        <div style='font-family:sans-serif;background:#1a1a2e;padding:16px;border-radius:8px;margin-bottom:12px'>
          <h3 style='color:#f0c040;margin:0 0 4px 0'>📋 Pre-Submission Checklist</h3>
          <p style='color:#888;margin:0'>Paste your Devpost description and README. Get a full readiness report before you submit.</p>
        </div>
        """))
        display(HTML("<p style='font-family:sans-serif;font-weight:bold'>Devpost Description</p>"))
        display(cl_project_input)
        display(HTML("<p style='font-family:sans-serif;font-weight:bold'>README</p>"))
        display(cl_readme_input)
        display(cl_btn)
        display(cl_output)

tool1_btn.on_click(show_judge)
tool2_btn.on_click(show_checklist)

# Show the main menu
display(HTML("""
<div style='font-family:sans-serif;background:#1a1a2e;padding:24px;border-radius:12px;margin:10px 0'>
  <h2 style='color:#f0c040;margin-top:0'>🏆 HackMate</h2>
  <p style='color:#aaa;margin-bottom:16px'>Your AI copilot for hackathons. Pick a tool:</p>
  <p style='color:#cdd6f4;margin:6px 0'><b style='color:#4caf50'>1</b> &nbsp; Practice your pitch against an adaptive AI judge — get scored after 5 questions</p>
  <p style='color:#cdd6f4;margin:6px 0'><b style='color:#ff9800'>2</b> &nbsp; Review your Devpost + README before submitting — get a readiness report</p>
</div>
"""))
display(widgets.HBox([tool1_btn, tool2_btn], layout=widgets.Layout(gap="16px", margin="0 0 16px 0")))
display(main_output)

C:\Users\varun\AppData\Local\Temp\ipykernel_18716\4097413442.py:119: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  answer_input.on_submit(lambda x: on_send(None))


Output()

> 💡 **Start over?** Go to **Kernel → Restart & Run All**.